# 6. Critic Agent

The **CriticAgent** reviews outputs from Research and Execution agents. It checks for hallucinations, inconsistencies, and errors.

Output is a `CriticReport` with a pass/fail status and a list of issues per sub-task.

In [ ]:
from unittest.mock import AsyncMock, MagicMock
import json

from mares.agents.critic_agent import CriticAgent
from mares.models.agent_output import AgentOutput

mock_factory = MagicMock()
mock_factory.generate = AsyncMock()
mock_factory.generate.return_value = json.dumps({
    "passed": True,
    "issues": [],
    "summary": "All outputs look correct and well-sourced."
})

agent = CriticAgent(llm_factory=mock_factory)

# Create sample outputs to review
outputs = [
    AgentOutput(
        agent="research_agent",
        sub_task_id=1,
        content={"summary": "Python asyncio is great for I/O", "facts": ["async/await syntax"]},
        metadata={"sources": ["https://docs.python.org"]}
    ),
    AgentOutput(
        agent="execution_agent",
        sub_task_id=2,
        content={"language": "python", "code": "print('ok')", "success": True},
    ),
]

report = await agent.run(outputs)
print(f"Passed: {report.passed}")
print(f"Issues: {report.issues}")
print(f"Summary: {report.summary}")

## Failure Detection

When the critic finds issues, it lists them with severity levels:

In [ ]:
# Simulate a critic finding problems
mock_factory.generate.return_value = json.dumps({
    "passed": False,
    "issues": [
        {"sub_task_id": 1, "severity": "high", "description": "Claim lacks supporting source"},
        {"sub_task_id": 2, "severity": "medium", "description": "Code doesn't handle edge case"}
    ],
    "summary": "Two issues found that need correction."
})

report2 = await agent.run(outputs)
print(f"Passed: {report2.passed}")
for issue in report2.issues:
    print(f"  [{issue['severity']}] Task #{issue['sub_task_id']}: {issue['description']}")
print(f"Failed sub-tasks: {report2.failed_subtask_ids()}")